# 🌲 Çam-AI × YOLO — ortomozaikte çam tespiti (Google Colab)

**Tez:** İHA (DJI P1 + Pix4D) ortomozaiğinde çam ağaçlarının derin öğrenme ile tespiti — YTÜ Harita Mühendisliği.

Bu defter, Çam-AI uygulamasının (**kopru.html → Adım 5 → 📦 YOLO veri seti (ZIP)**) ürettiği veri setiyle
bir **YOLO (Ultralytics YOLO11 / YOLOv8)** nesne tespit modeli eğitir, doğrulama metriklerini üretir,
**tüm ortomozaik karolarında** tahmin yapar ve sonuçları **`yolo_tespitler.geojson`** olarak verir.
Bu dosyayı Çam-AI'de **Doğruluk Analizi → 📥 Dış tespitleri yükle** ile açıp, kendi modelinizle aynı
referans ve aynı protokolle (taç poligonu içine ≥1 tespit) puanlarsınız. Böylece tezde iki yöntemi
(öznitelik tabanlı MLP ve YOLO) **aynı test çamları** üzerinde karşılaştırırsınız.

**Gerekenler**
1. Çam-AI'den indirdiğiniz `cam_yolo_veri_….zip` (Google Drive'a yükleyin; büyük dosya için en hızlısı budur).
2. Colab'da GPU: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU**.

**Süre (T4):** kurulum ~2 dk · eğitim 150 epoch yolo11m @1024 px için ~30–60 dk (veri boyutuna göre) · tahmin 1–5 dk.

> Hücreleri **sırayla** çalıştırın (Shift+Enter). Her hücrenin üstündeki açıklama ne yaptığını anlatır.

## 1. GPU denetimi ve kurulum

In [ ]:
# GPU görünmeli (Tesla T4 vb.). Görünmüyorsa: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU
!nvidia-smi -L || echo "⚠️ GPU yok — eğitim çok yavaş olur"

# Ultralytics (YOLO11 / YOLOv8). Sürüm 8.3 ve üstü YOLO11 ağırlıklarını içerir.
!pip install -q "ultralytics>=8.3.100,<8.4"

import ultralytics, torch
print("ultralytics", ultralytics.__version__, "· torch", torch.__version__, "· CUDA:", torch.cuda.is_available())

## 2. Veri setini getir

**Seçenek A (önerilen):** ZIP'i Google Drive'ınıza yükleyin ve yolunu `ZIP_YOLU` değişkenine yazın
(ör. `/content/drive/MyDrive/cam_yolo_veri_2026-10-14_1530.zip`). Drive bağlantısı için izin penceresi açılır.

**Seçenek B:** `ZIP_YOLU` boş bırakılırsa dosya seçme penceresi açılır ve ZIP bilgisayardan yüklenir (yavaş olabilir).

In [ ]:
import os, zipfile, glob, shutil, json

ZIP_YOLU = ""   # ← Drive'daki ZIP yolu; boş ise bilgisayardan yükleme penceresi açılır
VERI = "/content/veri"

if ZIP_YOLU.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
if not ZIP_YOLU:
    from google.colab import files
    yuklenen = files.upload()                       # tarayıcıda dosya seç
    ZIP_YOLU = "/content/" + list(yuklenen.keys())[0]
assert os.path.exists(ZIP_YOLU), "ZIP bulunamadı: " + ZIP_YOLU

shutil.rmtree(VERI, ignore_errors=True)
os.makedirs(VERI, exist_ok=True)
with zipfile.ZipFile(ZIP_YOLU) as z:
    z.extractall(VERI)

# data.yaml içindeki göreli yolu mutlak yola çevir (Ultralytics 'path:' değerini kendi veri klasörüne göre çözer)
with open(os.path.join(VERI, "data.yaml"), "w", encoding="utf-8") as f:
    f.write(f"path: {VERI}\ntrain: images/train\nval: images/val\nnames:\n  0: cam\n")

print(open(os.path.join(VERI, "README_tr.txt"), encoding="utf-8").read())
meta = json.load(open(os.path.join(VERI, "tahmin", "karolar.json"), encoding="utf-8"))["meta"]
print("\nMeta:", {k: meta[k] for k in ("epsg", "sx", "calisma_w", "calisma_h", "karo", "bindirme", "egitim_tac", "test_tac", "kume")})

### 2b. (İsteğe bağlı) Roboflow'da etiketlediğiniz fotoğrafları ekleyin

Roboflow'dan **YOLOv8 formatında** indirdiğiniz ZIP'in yolunu `EK_VERI_ZIP`'e yazın. İçindeki `train/` ve `test/`
görüntüleri eğitime, `valid/` görüntüleri doğrulamaya eklenir; yalnız **cam** sınıfı alınır (başka sınıf varsa atılır).
Boş bırakırsanız bu hücre hiçbir şey yapmaz.

> ⚠️ Sızıntı uyarısı: ham fotoğraflarda **test çamları** da görünebilir. Bu durumda YOLO test çamlarını eğitimde görmüş olur ve
> Çam-AI'deki karşılaştırma adil olmaz. Tezde bu modeli "ortomozaik + fotoğraf" varyantı olarak ayrıca raporlayın;
> temiz karşılaştırma için bu hücreyi boş bırakıp yalnız ortomozaik karolarıyla eğitilen modeli kullanın.

In [ ]:
EK_VERI_ZIP = ""   # ör. "/content/drive/MyDrive/cam.v3i.yolov8.zip" (Roboflow → Export → YOLOv8)

if EK_VERI_ZIP:
    import yaml
    EK = "/content/ek_veri"
    shutil.rmtree(EK, ignore_errors=True); os.makedirs(EK)
    with zipfile.ZipFile(EK_VERI_ZIP) as z:
        z.extractall(EK)
    # sınıf adları: 'cam' (ya da tek sınıf) → 0
    ek_yaml = glob.glob(f"{EK}/**/data.yaml", recursive=True)
    adlar = yaml.safe_load(open(ek_yaml[0]))["names"] if ek_yaml else ["cam"]
    if isinstance(adlar, dict): adlar = [adlar[k] for k in sorted(adlar)]
    cam_idx = {i for i, a in enumerate(adlar) if str(a).lower().strip() in ("cam", "çam", "pine", "pinus")} or ({0} if len(adlar) == 1 else set())
    print("Ek veri sınıfları:", adlar, "→ alınan indeksler:", sorted(cam_idx))
    eklenen = {"train": 0, "val": 0}; kutu = 0; atlanan = 0
    for bolum, hedef in (("train", "train"), ("test", "train"), ("valid", "val"), ("val", "val")):
        for img in glob.glob(f"{EK}/**/{bolum}/images/*", recursive=True):
            kok = os.path.splitext(os.path.basename(img))[0]
            etiket = os.path.join(os.path.dirname(os.path.dirname(img)), "labels", kok + ".txt")
            satirlar = []
            if os.path.exists(etiket):
                for s in open(etiket).read().splitlines():
                    p = s.split()
                    if len(p) >= 5 and int(float(p[0])) in cam_idx:
                        satirlar.append("0 " + " ".join(p[1:5]))
            if not satirlar:
                atlanan += 1; continue                      # kutusuz fotoğraf: eklenmez
            yeni = "rf_" + kok.replace(".", "_")
            shutil.copy(img, f"{VERI}/images/{hedef}/{yeni}{os.path.splitext(img)[1].lower()}")
            open(f"{VERI}/labels/{hedef}/{yeni}.txt", "w").write("\n".join(satirlar))
            eklenen[hedef] += 1; kutu += len(satirlar)
    print(f"Eklendi: eğitim {eklenen['train']} · doğrulama {eklenen['val']} görüntü, {kutu} çam kutusu; {atlanan} kutusuz görüntü atlandı")
else:
    print("Ek veri yok; yalnız ortomozaik karolarıyla devam.")

### 2c. Pozitif karoları çoğalt (küçük veri seti için gerekli)

Eğitimde yalnız ~16 kutulu karo varsa bir epoch 4 adımdan ibaret kalır; model yüzlerce adımda hiçbir şey öğrenemez
(mAP ≈ 0). Bu hücre kutulu eğitim karolarını `COGALT` kat kopyalar. Kopyalar birebir aynı dosyadır ama her epoch'ta
**farklı rastgele artırma** (çevirme, döndürme, ölçek, mozaik, renk) aldıkları için modele farklı örnekler gibi görünür.
Doğrulama kümesine dokunulmaz.

In [ ]:
COGALT = 6   # kutulu eğitim karosu başına kopya sayısı (toplam = COGALT kat)

import re
for p in glob.glob(f"{VERI}/images/train/*"):
    kok, uz = os.path.splitext(os.path.basename(p))
    if re.search(r"_k\d+$", kok):               # daha önce üretilmiş kopya
        continue
    etiket = f"{VERI}/labels/train/{kok}.txt"
    if not (os.path.exists(etiket) and open(etiket).read().strip()):
        continue                                 # arka plan karosu çoğaltılmaz
    for k in range(1, COGALT):
        hedef = f"{VERI}/images/train/{kok}_k{k}{uz}"
        if not os.path.exists(hedef):
            shutil.copy(p, hedef)
            shutil.copy(etiket, f"{VERI}/labels/train/{kok}_k{k}.txt")
for c in glob.glob(f"{VERI}/labels/train*.cache"):
    os.remove(c)                                 # eski önbellek yeni dosyaları görmesin
poz = sum(1 for l in glob.glob(f"{VERI}/labels/train/*.txt") if open(l).read().strip())
print(f"Eğitim kümesi: {len(glob.glob(f'{VERI}/images/train/*'))} görüntü, {poz} kutulu (çoğaltma dahil)")

## 3. Veri özeti ve örnek karolar
Kutuların taçların üzerine oturduğunu gözle doğrulayın (eğitim taçları kutulu; test taçları düz renkle maskelidir).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def say(bolum):
    g = sorted(glob.glob(f"{VERI}/images/{bolum}/*"))
    e = [p for p in g if open(f"{VERI}/labels/{bolum}/" + os.path.splitext(os.path.basename(p))[0] + ".txt").read().strip()]
    return len(g), len(e)
for b in ("train", "val"):
    n, p = say(b)
    print(f"{b}: {n} görüntü ({p} kutulu, {n-p} arka plan)")
print("tahmin karoları:", len(glob.glob(f"{VERI}/tahmin/images/*.jpg")))

ornekler = [p for p in sorted(glob.glob(f"{VERI}/images/train/*.jpg"))
            if open(f"{VERI}/labels/train/" + os.path.basename(p)[:-4] + ".txt").read().strip()][:4]
fig, eks = plt.subplots(1, max(1, len(ornekler)), figsize=(5 * max(1, len(ornekler)), 5))
for ax, p in zip(eks if len(ornekler) > 1 else [eks], ornekler):
    im = Image.open(p).convert("RGB"); W, H = im.size; cz = ImageDraw.Draw(im)
    for s in open(f"{VERI}/labels/train/" + os.path.basename(p)[:-4] + ".txt").read().split("\n"):
        if not s.strip(): continue
        _, cx, cy, w, h = map(float, s.split())
        cz.rectangle([(cx - w/2) * W, (cy - h/2) * H, (cx + w/2) * W, (cy + h/2) * H], outline=(255, 230, 0), width=4)
    ax.imshow(im); ax.set_title(os.path.basename(p), fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Eğitim

| Model | Parametre | T4'te 1024 px için parti | Not |
|---|---|---|---|
| **`yolo11s.pt`** (varsayılan) | 9 M | 8–16 | küçük veri setinde en kararlı; tez için önerilen başlangıç |
| `yolo11m.pt` | 20 M | 8 | daha güçlü ama az veriyle zor öğrenir; veri artınca deneyin |
| `yolo11l.pt` | 25 M | 4 | daha yavaş, küçük veri setinde kazanç belirsiz |
| `yolov8m.pt` | 26 M | 8 | referans tezle (YOLOv4/v5) kıyas için eski nesil |

Önemli ayarlar: `imgsz=1024` (karo boyu), `epochs=120`, erken durdurma **kapalı** (`patience=0`; 6 nesnelik doğrulama kümesi
erken durdurmayı yanıltır, en iyi ağırlık yine `best.pt` olarak seçilir), AdamW `lr0=0.001` + 5 epoch ısınma (küçük veride
kararlılık), COCO ön eğitimli ağırlık (transfer öğrenme), havadan görüntüye uygun artırmalar (`flipud`, `fliplr`, `degrees`),
tohum 42 (tekrarlanabilirlik). **CUDA out of memory** alırsanız `BATCH`'i yarıya indirin.

In [ ]:
from ultralytics import YOLO

MODEL    = "yolo11s.pt"   # yolo11m.pt / yolo11l.pt / yolov8m.pt
EPOCHS   = 120
IMGSZ    = 1024
BATCH    = 8
PATIENCE = 0              # 0 = erken durdurma kapalı
CALISMA  = "/content/calisma"
AD       = "cam_" + MODEL.split(".")[0]

model = YOLO(MODEL)
sonuc = model.train(
    data=f"{VERI}/data.yaml", imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
    project=CALISMA, name=AD, exist_ok=True, seed=42, deterministic=True,
    optimizer="AdamW", lr0=0.001, lrf=0.05, warmup_epochs=5, cos_lr=True, close_mosaic=10,
    flipud=0.5, fliplr=0.5, degrees=10.0, scale=0.3, mosaic=1.0, mixup=0.0,
    plots=True, verbose=True)
EN_IYI = f"{CALISMA}/{AD}/weights/best.pt"
print("En iyi ağırlık:", EN_IYI)

## 5. Doğrulama metrikleri (tez tablosu için)

Doğrulama kümesi = pozitif karoların %20'si (eğitim taçlarının bir kısmı). **Bağımsız test çamları** burada
değil, Çam-AI'deki değerlendirmede ölçülür; tezde ikisini de ayrı ayrı raporlayın.

In [ ]:
from IPython.display import Image as GImage, display
import pandas as pd

model = YOLO(EN_IYI)
val = model.val(data=f"{VERI}/data.yaml", imgsz=IMGSZ, batch=BATCH, split="val", plots=True,
                project=CALISMA, name=AD + "_val", exist_ok=True, verbose=False)
P, R = float(val.box.mp), float(val.box.mr)
F1 = 2 * P * R / (P + R) if P + R else 0.0
print(f"Doğrulama kümesi: Precision {P:.3f} · Recall {R:.3f} · F1 {F1:.3f} · mAP50 {float(val.box.map50):.3f} · mAP50-95 {float(val.box.map):.3f}")

egitim_csv = pd.read_csv(f"{CALISMA}/{AD}/results.csv")
egitim_csv.columns = [c.strip() for c in egitim_csv.columns]
print("Epoch sayısı:", len(egitim_csv), "· son epoch mAP50:", round(float(egitim_csv['metrics/mAP50(B)'].iloc[-1]), 3))
for g in ("results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"):
    yol = f"{CALISMA}/{AD}/{g}"
    if os.path.exists(yol):
        print("\n", g); display(GImage(filename=yol, width=700))

## 6. Ortomozaik üzerinde tahmin ve coğrafi birleştirme

Model, ZIP'teki **maskesiz tahmin karolarının tümünde** çalıştırılır (`conf=0.05`: düşük eşik, çünkü en iyi
güven eşiği Çam-AI'deki 🔎 taramayla referansa göre seçilir). Karolar %20 bindirmeli olduğu için aynı ağaç
iki karoda çıkabilir; aşağıdaki birleştirici kutuları ortomozaik pikseline taşır, **küresel NMS** ile tekilleştirir
ve `karolar.json` içindeki dönüşümle (E = ox + px·sx, N = oy − py·sy, EPSG meta'da) GeoJSON yazar.

In [ ]:
# --- Çam-AI birleştirici modülü (cpp/yolo/yolo_tespit_birlestir.py ile aynı) ---
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Çam-AI ↔ YOLO köprüsü: karo bazlı YOLO tespitlerini ortomozaik koordinatına taşır.

Girdi
  • veri klasörü  : Çam-AI'nin ürettiği ZIP'in açılmış hâli (tahmin/karolar.json burada)
  • tahmin dosyası: [{"dosya": "karo_00000_00000.jpg", "kutular": [[x1,y1,x2,y2,guven], ...]}, ...]
                    (kutu köşeleri karo pikseli, 0..karo boyu)
Çıktı (cikti klasörü)
  • yolo_tespitler.geojson : ağaç başına TEK NOKTA (kutu merkezi) + "guven" özelliği → Çam-AI "Dış tespitleri yükle"
  • yolo_kutular.geojson   : aynı tespitlerin kutu poligonları (QGIS'te bakmak için)
  • yolo_ozet.txt          : eşik başına tespit sayısı

Yalnız standart kütüphane kullanır (numpy gerekmez); Colab defteri bu dosyayı gömülü olarak taşır.
"""
import argparse
import json
import math
import os


def karolari_oku(veri_klasoru):
    yol = os.path.join(veri_klasoru, 'tahmin', 'karolar.json')
    with open(yol, encoding='utf-8') as f:
        k = json.load(f)
    return k['meta'], {c['dosya']: c for c in k['karolar']}


def kutulari_calisma_pikseline_tasi(tahminler, karolar, karo_boyu):
    """Karo koordinatındaki kutuları çalışma (ortomozaik) pikseline çevirir; dolgu bölgesine
    düşenleri (karonun gerçek eninden/boyundan dışarıda kalan merkezler) atar."""
    liste = []
    for t in tahminler:
        karo = karolar.get(t['dosya']) or karolar.get(os.path.basename(t['dosya']))
        if karo is None:
            continue
        for k in t.get('kutular', []):
            if len(k) < 5:
                continue
            x1, y1, x2, y2, p = [float(v) for v in k[:5]]
            if x2 <= x1 or y2 <= y1:
                continue
            mx, my = (x1 + x2) / 2, (y1 + y2) / 2
            if mx > karo['w'] or my > karo['h']:          # siyah dolgu
                continue
            x1, x2 = max(0.0, x1), min(float(karo['w']), x2)
            y1, y2 = max(0.0, y1), min(float(karo['h']), y2)
            if x2 - x1 < 2 or y2 - y1 < 2:
                continue
            # karo kenarına değen kutu: ağaç karo sınırında kesilmiş olabilir (bindirme kopyası adayı)
            kenar = x1 <= 1.5 or y1 <= 1.5 or x2 >= karo['w'] - 1.5 or y2 >= karo['h'] - 1.5
            liste.append({'x1': karo['x0'] + x1, 'y1': karo['y0'] + y1,
                          'x2': karo['x0'] + x2, 'y2': karo['y0'] + y2,
                          'p': max(0.0, min(1.0, p)), 'karo': karo['dosya'], 'kenar': kenar})
    return liste


def _iou(a, b):
    kx = max(0.0, min(a['x2'], b['x2']) - max(a['x1'], b['x1']))
    ky = max(0.0, min(a['y2'], b['y2']) - max(a['y1'], b['y1']))
    kes = kx * ky
    if kes <= 0:
        return 0.0
    aa = (a['x2'] - a['x1']) * (a['y2'] - a['y1'])
    ab = (b['x2'] - b['x1']) * (b['y2'] - b['y1'])
    return kes / (aa + ab - kes)


def _merkez_icinde(a, b):
    """b'nin merkezi a'nın içinde mi (bindirme sınırında yarım kalan kutular için)."""
    mx, my = (b['x1'] + b['x2']) / 2, (b['y1'] + b['y2']) / 2
    return a['x1'] <= mx <= a['x2'] and a['y1'] <= my <= a['y2']


def _kesisim_kucuk_orani(a, b):
    """Kesişim / küçük kutunun alanı (IoS). Karo sınırında ikiye bölünmüş aynı ağacın iki yarım kutusu
    IoU'da düşük kalır ama bindirme kuşağı kadar kesiştikleri için IoS yüksektir."""
    kx = max(0.0, min(a['x2'], b['x2']) - max(a['x1'], b['x1']))
    ky = max(0.0, min(a['y2'], b['y2']) - max(a['y1'], b['y1']))
    kes = kx * ky
    if kes <= 0:
        return 0.0
    kucuk = min((a['x2'] - a['x1']) * (a['y2'] - a['y1']), (b['x2'] - b['x1']) * (b['y2'] - b['y1']))
    return kes / kucuk if kucuk > 0 else 0.0


def kuresel_nms(kutular, iou_esik=0.45, ios_esik=0.5):
    """Güvene göre sıralı, tüm ortomozaik üzerinde bastırma. Bir kutu şu durumlarda elenir:
       • daha güvenli bir kutuyla IoU ≥ iou_esik (klasik NMS),
       • merkezi daha güvenli ve daha büyük bir kutunun içinde (karo kenarında yarım kalan kopya),
       • FARKLI karolardan gelen ve ikisi de karo kenarına değen iki kutunun kesişimi küçüğünün ≥ ios_esik'i
         (karo sınırında ikiye bölünmüş, bindirmeden büyük taç); tutulan kutu iki parçanın birleşimine
         genişletilir ki nokta taç merkezine düşsün. Aynı karodaki bitişik iki gerçek ağaç bu kurala girmez.
    Bindirmeli karolardan gelen kopyaları tek tespite indirger."""
    sirali = sorted(kutular, key=lambda k: -k['p'])
    tutulan = []
    for k in sirali:
        alan_k = (k['x2'] - k['x1']) * (k['y2'] - k['y1'])
        bastir = False
        for t in tutulan:
            if _iou(t, k) >= iou_esik:
                bastir = True
                break
            alan_t = (t['x2'] - t['x1']) * (t['y2'] - t['y1'])
            if alan_k <= alan_t and _merkez_icinde(t, k):
                bastir = True
                break
            if (t.get('kenar') and k.get('kenar') and t.get('karo') != k.get('karo')
                    and _kesisim_kucuk_orani(t, k) >= ios_esik):
                t['x1'], t['y1'] = min(t['x1'], k['x1']), min(t['y1'], k['y1'])
                t['x2'], t['y2'] = max(t['x2'], k['x2']), max(t['y2'], k['y2'])
                bastir = True
                break
        if not bastir:
            tutulan.append(dict(k))
    return tutulan


def cografi(meta, px, py):
    """Çalışma pikseli → (Doğu, Kuzey). Coğrafi bilgi yoksa özgün görüntü pikseli (y negatif) döner;
    Çam-AI bu biçimi de okur."""
    if meta.get('epsg') and meta.get('sx'):
        return meta['ox'] + px * meta['sx'], meta['oy'] - py * meta['sy']
    o = meta.get('olcek') or 1
    return px * o, -py * o


def geojson_yaz(kutular, meta, cikti_klasoru):
    os.makedirs(cikti_klasoru, exist_ok=True)
    crs = ({'type': 'name', 'properties': {'name': 'urn:ogc:def:crs:EPSG::%d' % int(meta['epsg'])}}
           if meta.get('epsg') else None)
    sx = meta.get('sx') or meta.get('olcek') or 1
    sy = meta.get('sy') or meta.get('olcek') or 1
    noktalar, poligonlar = [], []
    for i, k in enumerate(sorted(kutular, key=lambda k: -k['p']), start=1):
        mx, my = (k['x1'] + k['x2']) / 2, (k['y1'] + k['y2']) / 2
        e, n = cografi(meta, mx, my)
        oz = {'fid': i, 'guven': round(k['p'], 4), 'sinif': 'cam',
              'kutu_en': round((k['x2'] - k['x1']) * sx, 2), 'kutu_boy': round((k['y2'] - k['y1']) * sy, 2),
              'karo': k['karo']}
        noktalar.append({'type': 'Feature', 'properties': oz,
                         'geometry': {'type': 'Point', 'coordinates': [round(e, 3), round(n, 3)]}})
        kose = [cografi(meta, k['x1'], k['y1']), cografi(meta, k['x2'], k['y1']),
                cografi(meta, k['x2'], k['y2']), cografi(meta, k['x1'], k['y2'])]
        kose = [[round(a, 3), round(b, 3)] for a, b in kose]
        poligonlar.append({'type': 'Feature', 'properties': dict(oz),
                           'geometry': {'type': 'Polygon', 'coordinates': [kose + [kose[0]]]}})
    for ad, ozellikler in (('yolo_tespitler.geojson', noktalar), ('yolo_kutular.geojson', poligonlar)):
        fc = {'type': 'FeatureCollection', 'features': ozellikler}
        if crs:
            fc['crs'] = crs
        with open(os.path.join(cikti_klasoru, ad), 'w', encoding='utf-8') as f:
            json.dump(fc, f, ensure_ascii=False)
    satirlar = ['YOLO tespit özeti — kaynak: %s' % meta.get('kaynak', '?'),
                'Toplam (NMS sonrası): %d' % len(kutular), 'Güven eşiği → tespit sayısı:']
    for e in range(5, 100, 5):
        satirlar.append('  ≥ %%%2d : %d' % (e, sum(1 for k in kutular if k['p'] >= e / 100 - 1e-9)))
    with open(os.path.join(cikti_klasoru, 'yolo_ozet.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(satirlar) + '\n')
    return satirlar


def birlestir(veri_klasoru, tahminler, cikti_klasoru, nms_iou=0.45):
    """Tam akış: karo kutuları → çalışma pikseli → küresel NMS → GeoJSON. Tutulan kutuları döndürür."""
    meta, karolar = karolari_oku(veri_klasoru)
    ham = kutulari_calisma_pikseline_tasi(tahminler, karolar, meta.get('karo', 1024))
    tutulan = kuresel_nms(ham, nms_iou)
    ozet = geojson_yaz(tutulan, meta, cikti_klasoru)
    print('Ham kutu: %d → NMS sonrası: %d' % (len(ham), len(tutulan)))
    print('\n'.join(ozet))
    return tutulan


def main():
    ap = argparse.ArgumentParser(description='YOLO karo tespitlerini ortomozaik GeoJSON\'una çevirir')
    ap.add_argument('--veri', required=True, help='açılmış Çam-AI YOLO veri klasörü')
    ap.add_argument('--tahmin', required=True, help='karo tahminleri JSON')
    ap.add_argument('--cikti', required=True, help='çıktı klasörü')
    ap.add_argument('--nms', type=float, default=0.45, help='küresel NMS IoU eşiği')
    a = ap.parse_args()
    with open(a.tahmin, encoding='utf-8') as f:
        tahminler = json.load(f)
    birlestir(a.veri, tahminler, a.cikti, a.nms)

In [ ]:
import time
CIKTI = "/content/cikti"
os.makedirs(CIKTI, exist_ok=True)

t0 = time.time()
tahminler = []
for r in model.predict(source=f"{VERI}/tahmin/images", conf=0.05, iou=0.6, imgsz=IMGSZ,
                       stream=True, verbose=False, max_det=500, half=True):
    kutular = r.boxes.xyxy.cpu().tolist()
    guven = r.boxes.conf.cpu().tolist()
    tahminler.append({"dosya": os.path.basename(r.path), "kutular": [k + [p] for k, p in zip(kutular, guven)]})
print(f"{len(tahminler)} karo tahmin edildi ({time.time() - t0:.0f} sn)")
json.dump(tahminler, open(f"{CIKTI}/karo_tahminleri.json", "w"))

tutulan = birlestir(VERI, tahminler, CIKTI, nms_iou=0.45)

### Hızlı görsel denetim
En yüksek güvenli birkaç tespit, karo üzerinde (yeşil kutu = YOLO).

In [ ]:
karolar_meta = json.load(open(f"{VERI}/tahmin/karolar.json"))
karo_sozluk = {k["dosya"]: k for k in karolar_meta["karolar"]}
en_iyi_karolar = {}
for t in sorted(tutulan, key=lambda k: -k["p"])[:200]:
    en_iyi_karolar.setdefault(t["karo"], []).append(t)
secilen = sorted(en_iyi_karolar.items(), key=lambda kv: -len(kv[1]))[:3]
fig, eks = plt.subplots(1, max(1, len(secilen)), figsize=(6 * max(1, len(secilen)), 6))
for ax, (dosya, liste) in zip(eks if len(secilen) > 1 else [eks], secilen):
    im = Image.open(f"{VERI}/tahmin/images/{dosya}").convert("RGB"); cz = ImageDraw.Draw(im); k = karo_sozluk[dosya]
    for t in liste:
        cz.rectangle([t["x1"] - k["x0"], t["y1"] - k["y0"], t["x2"] - k["x0"], t["y2"] - k["y0"]], outline=(0, 255, 90), width=4)
        cz.text((t["x1"] - k["x0"] + 4, t["y1"] - k["y0"] + 4), f"{t['p']:.2f}", fill=(255, 255, 0))
    ax.imshow(im); ax.set_title(dosya, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Sonuçları indir

`yolo_sonuclar_<model>.zip` içinde: `yolo_tespitler.geojson` (Çam-AI'ye yüklenecek nokta dosyası), `yolo_kutular.geojson`
(QGIS'te kutular), `yolo_ozet.txt`, `best.pt` (eğitilmiş ağırlık), `results.csv/png`, karışıklık matrisi ve PR eğrisi,
`args.yaml` (tüm hiperparametreler — tez yöntem bölümü için).

In [ ]:
SONUC_ZIP = f"/content/yolo_sonuclar_{AD}.zip"
with zipfile.ZipFile(SONUC_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for ad in ("yolo_tespitler.geojson", "yolo_kutular.geojson", "yolo_ozet.txt", "karo_tahminleri.json"):
        z.write(f"{CIKTI}/{ad}", ad)
    z.write(EN_IYI, "best.pt")
    for ad in ("results.csv", "results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png", "args.yaml"):
        yol = f"{CALISMA}/{AD}/{ad}"
        if os.path.exists(yol): z.write(yol, "egitim/" + ad)
    with open(f"{CIKTI}/dogrulama_metrikleri.txt", "w", encoding="utf-8") as f:
        f.write(f"model: {MODEL}\nimgsz: {IMGSZ}\nepochs(max): {EPOCHS}\nbatch: {BATCH}\n"
                f"val Precision: {P:.4f}\nval Recall: {R:.4f}\nval F1: {F1:.4f}\nval mAP50: {float(val.box.map50):.4f}\nval mAP50-95: {float(val.box.map):.4f}\n")
    z.write(f"{CIKTI}/dogrulama_metrikleri.txt", "dogrulama_metrikleri.txt")
print("Hazır:", SONUC_ZIP, round(os.path.getsize(SONUC_ZIP) / 1048576, 1), "MB")

try:
    from google.colab import files
    files.download(SONUC_ZIP)
except Exception as e:
    print("Otomatik indirme yapılamadı:", e)
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy(SONUC_ZIP, "/content/drive/MyDrive/"); print("Drive'a da kopyalandı: MyDrive/" + os.path.basename(SONUC_ZIP))

## 8. Çam-AI'de değerlendirme ve tezde raporlama

1. **kopru.html**'de ortomozaiği (aynı çözünürlükle), taç/nokta/küme GeoJSON'larını ve 🎓 eğitim kimliklerini yükleyin
   (ZIP'i üretirken kullandığınız ayarlarla aynı olmalı; `README_tr.txt` kimlikleri listeler).
2. Doğruluk Analizi → **📥 Dış tespitleri yükle** → `yolo_tespitler.geojson`. Eşleşme protokolü: **Taç poligonu**.
3. **🔎 Ayar tara** (yalnız güven eşiği taranır) → **Değerlendir** → **📄 Doğruluk raporu (TXT)**.
4. Aynı adımları kendi Çam-AI modeliniz için **📍 Ağaçları İşaretle** ile tekrarlayın; iki rapor aynı test çamlarını puanlar.

Tez için önerilen tablo:

| Yöntem | Eğitim verisi | Test birimi | TP | FP | FN | Precision | Recall | F1 |
|---|---|---|---|---|---|---|---|---|
| Çam-AI (36 öznitelik + MLP) | eğitim taçları (ortomozaik) | test taçları + kümeler | | | | | | |
| YOLO11m (bu defter) | aynı eğitim taçları | aynı test taçları + kümeler | | | | | | |

Ek olarak raporlayın: doğrulama kümesi mAP50 / F1 (5. bölüm), epoch sayısı ve süre, `args.yaml`'daki hiperparametreler,
güven eşiği taraması tablosu (rapor TXT'de). Referans tezde (Öztürk, 2022) YOLOv4/v5 ile F1 0,53–0,89 aralığı bildirilmiştir;
sonuçlarınızı bu bağlamda tartışabilirsiniz.

## Ek — birden fazla modeli aynı veriyle karşılaştırma (isteğe bağlı)

`KARSILASTIR = True` yapıp çalıştırırsanız listedeki her model aynı ayarlarla eğitilir ve doğrulama metrikleri tek tabloda
toplanır (her model T4'te onlarca dakika sürer). Tez için sonra en iyisini 4–7. bölümlerle çalıştırın (`MODEL` değişkenini değiştirin).

In [ ]:
KARSILASTIR = False
MODELLER = ["yolo11s.pt", "yolo11m.pt", "yolov8m.pt"]
if KARSILASTIR:
    satirlar = []
    for m in MODELLER:
        ad = "kiyas_" + m.split(".")[0]
        YOLO(m).train(data=f"{VERI}/data.yaml", imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
                      project=CALISMA, name=ad, exist_ok=True, seed=42, deterministic=True,
                      optimizer="AdamW", lr0=0.001, lrf=0.05, warmup_epochs=5, cos_lr=True, close_mosaic=10,
                      flipud=0.5, fliplr=0.5, degrees=10.0, scale=0.3, plots=False, verbose=False)
        v = YOLO(f"{CALISMA}/{ad}/weights/best.pt").val(data=f"{VERI}/data.yaml", imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
        p, r = float(v.box.mp), float(v.box.mr)
        satirlar.append({"model": m, "P": round(p, 3), "R": round(r, 3), "F1": round(2*p*r/(p+r) if p+r else 0, 3),
                         "mAP50": round(float(v.box.map50), 3), "mAP50-95": round(float(v.box.map), 3)})
    display(pd.DataFrame(satirlar))